# Extrair Clientes — RPA

**Projeto:** Assistente de Investimentos com RPA + n8n

**Fluxo:** `RPA/docs/index.html` → Python + BeautifulSoup → `clientes` → POST para o Webhook do n8n → n8n lê `RPA/docs/data.csv` → cruza perfil × saldo → Gmail.

**Arquivos**

| Arquivo | Papel |
|---|---|
| `RPA/docs/index.html` | Tabela de clientes (10) — trecho na próxima célula |
| `RPA/docs/data.csv` | Produtos por perfil (9) |
| `RPA/rpa/extrair-clientes.ipynb` | Este notebook |
| `RPA/rpa/rpa-workflow.json` | Workflow do n8n |

**Como rodar**

1. Terminal 1: `npx n8n start` → http://localhost:5678
2. Terminal 2: `cd RPA/docs` → `python -m http.server 8000`
3. No n8n: abra o workflow e clique em **Execute** (deixa em *Listening*)
4. Aqui: rode as células em ordem (Shift + Enter).


## 1. Estrutura da página HTML — trecho que o RPA lê

Fonte: arquivo local `RPA/docs/index.html` (não é link externo).

O RPA não lê o HTML inteiro — ele lê só a tabela `#clientes`. Trecho real do arquivo:

```html
<table id="clientes">
  <thead>
    <tr><th>Nome</th><th>Email</th><th>Saldo</th><th>Perfil</th></tr>
  </thead>
  <tbody>
    <tr>
      <td class="nome">Ana Silva</td>
      <td class="email">ana@email.com</td>
      <td class="saldo">R$ 12.500,00</td>
      <td class="perfil">Conservador</td>
    </tr>
    <!-- + 9 linhas, mesmo padrão, total 10 clientes -->
  </tbody>
</table>
```

Seletor usado no código: `#clientes tbody tr` — lê **todas** as linhas, sem número fixo.


## 2. Configuração

In [36]:
"""Configuração e localização dos arquivos locais."""
from pathlib import Path


def localizar(nome_arquivo: str) -> Path:
    """Acha RPA/docs/<arquivo> independente de onde o notebook foi aberto.

    O VS Code pode abrir o notebook com o cwd em RPA/rpa/ ou na raiz do
    repo. Por isso testamos os dois caminhos, em vez de um só.
    """
    candidatos = [
        Path("RPA/docs") / nome_arquivo,
        Path("../docs") / nome_arquivo,
        Path("docs") / nome_arquivo,
    ]
    for candidato in candidatos:
        if candidato.exists():
            return candidato
    raise FileNotFoundError(
        f"{nome_arquivo} não encontrado. Procurei em: {candidatos}. "
        "Confira se RPA/docs/ existe."
    )


HTML_CLIENTES = localizar("index.html")
CSV_PRODUTOS = localizar("data.csv")

# Webhook do n8n — modo TESTE.
# Só funciona enquanto o workflow estiver em "Listening" (botão Execute no n8n).
# Modo produção: http://localhost:5678/webhook/clientes (workflow Active)
N8N_WEBHOOK = "http://localhost:5678/webhook-test/clientes"

PERFIS_VALIDOS = ("Conservador", "Moderado", "Arrojado")

print(f"index.html -> {HTML_CLIENTES}")
print(f"data.csv   -> {CSV_PRODUTOS}")


index.html -> ..\docs\index.html
data.csv   -> ..\docs\data.csv


## 3. Extração

In [37]:
"""Extrai a lista de clientes de RPA/docs/index.html."""
import requests
from bs4 import BeautifulSoup


def extrair_clientes(caminho: Path) -> list[dict]:
    html = caminho.read_text(encoding="utf-8")
    clientes = []

    for linha in BeautifulSoup(html, "html.parser").select("#clientes tbody tr"):
        colunas = linha.find_all("td")
        if len(colunas) < 4:
            continue
        clientes.append(
            {
                "nome": colunas[0].get_text(strip=True),
                "email": colunas[1].get_text(strip=True),
                "saldo": colunas[2].get_text(strip=True),
                "perfil": colunas[3].get_text(strip=True),
            }
        )
    return clientes


clientes = extrair_clientes(HTML_CLIENTES)

print(f"{len(clientes)} clientes encontrados:")
for cliente in clientes:
    print(cliente)


10 clientes encontrados:
{'nome': 'Ana Silva', 'email': 'ana@email.com', 'saldo': 'R$ 12.500,00', 'perfil': 'Conservador'}
{'nome': 'Bruno Lima', 'email': 'bruno@email.com', 'saldo': 'R$ 3.200,00', 'perfil': 'Moderado'}
{'nome': 'Carla Souza', 'email': 'carla@email.com', 'saldo': 'R$ 28.900,00', 'perfil': 'Arrojado'}
{'nome': 'Daniel Oliveira', 'email': 'daniel@email.com', 'saldo': 'R$ 850,00', 'perfil': 'Conservador'}
{'nome': 'Elena Rodrigues', 'email': 'elena@email.com', 'saldo': 'R$ 45.000,00', 'perfil': 'Arrojado'}
{'nome': 'Felipe Santos', 'email': 'felipe@email.com', 'saldo': 'R$ 7.800,00', 'perfil': 'Moderado'}
{'nome': 'Gabriela Costa', 'email': 'gabriela@email.com', 'saldo': 'R$ 1.500,00', 'perfil': 'Conservador'}
{'nome': 'Henrique Alves', 'email': 'henrique@email.com', 'saldo': 'R$ 22.300,00', 'perfil': 'Moderado'}
{'nome': 'Isabela Martins', 'email': 'isabela@email.com', 'saldo': 'R$ 500,00', 'perfil': 'Conservador'}
{'nome': 'João Pereira', 'email': 'joao@email.com', 'sal

## 4. Validação

In [38]:
"""Valida os dados antes de enviar ao n8n."""
assert clientes, "Nenhum cliente encontrado em RPA/docs/index.html"
assert all("@" in c["email"] for c in clientes), "E-mail sem @ encontrado"
assert all(c["perfil"] in PERFIS_VALIDOS for c in clientes), "Perfil fora do esperado"
assert all(c["nome"] and c["saldo"] for c in clientes), "Campo vazio encontrado"

print(f"Validação OK — {len(clientes)} clientes prontos para envio.")


Validação OK — 10 clientes prontos para envio.


## 5. Prévia do cruze com data.csv (opcional)

In [39]:
"""Mostra qual produto cada cliente receberia. O cruze oficial é feito no n8n."""
import csv

produtos = list(csv.DictReader(CSV_PRODUTOS.open(encoding="utf-8")))


def para_numero(saldo: str) -> float:
    return float(saldo.replace("R$", "").replace(".", "").replace(",", ".").strip())


for cliente in clientes:
    elegiveis = [
        p["produto"]
        for p in produtos
        if p["perfil"] == cliente["perfil"]
        and para_numero(cliente["saldo"]) >= float(p["minimo"])
    ]
    print(f"{cliente['nome']} ({cliente['perfil']}, {cliente['saldo']}) -> {elegiveis}")


Ana Silva (Conservador, R$ 12.500,00) -> ['Poupança', 'CDB Liquidez Diária', 'Tesouro Selic']
Bruno Lima (Moderado, R$ 3.200,00) -> ['CDB Prefixado', 'Tesouro IPCA+']
Carla Souza (Arrojado, R$ 28.900,00) -> ['Ações ETF', 'Fundo de Ações', 'Criptomoedas']
Daniel Oliveira (Conservador, R$ 850,00) -> ['Poupança', 'Tesouro Selic']
Elena Rodrigues (Arrojado, R$ 45.000,00) -> ['Ações ETF', 'Fundo de Ações', 'Criptomoedas']
Felipe Santos (Moderado, R$ 7.800,00) -> ['CDB Prefixado', 'Fundo Multimercado', 'Tesouro IPCA+']
Gabriela Costa (Conservador, R$ 1.500,00) -> ['Poupança', 'CDB Liquidez Diária', 'Tesouro Selic']
Henrique Alves (Moderado, R$ 22.300,00) -> ['CDB Prefixado', 'Fundo Multimercado', 'Tesouro IPCA+']
Isabela Martins (Conservador, R$ 500,00) -> ['Poupança', 'Tesouro Selic']
João Pereira (Arrojado, R$ 15.000,00) -> ['Ações ETF', 'Fundo de Ações', 'Criptomoedas']


## 6. Envio ao n8n

In [40]:
"""Envia a lista para o Webhook do n8n.

Pré-requisito: no n8n, com o workflow aberto, clique em Execute e deixe
em "Waiting for test event" / Listening. Sem isso, o n8n devolve 404.
"""
resposta = requests.post(N8N_WEBHOOK, json={"clientes": clientes}, timeout=60)

print(f"POST {N8N_WEBHOOK} -> {resposta.status_code}")
print(resposta.text[:1000])

if resposta.status_code == 404:
    print("\nO n8n respondeu, mas nenhum workflow está escutando.")
    print("No n8n, clique em Execute (Listening) e rode esta célula novamente.")
elif resposta.status_code == 200:
    print("\nEnviado com sucesso.")


POST http://localhost:5678/webhook-test/clientes -> 500
{"message":"Error in workflow"}
